In [1]:
from moabb.datasets import PhysionetMI, Weibo2014
from moabb.paradigms import MotorImagery
from moabb.evaluations import WithinSessionEvaluation
from moabb.datasets.utils import find_intersecting_channels

from sklearn.pipeline import make_pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

from mne.decoding import CSP
from brainbot_dataset import get_brainbot_dataset
from datasets16 import PhysionetMI16, Weibo2014_16
from custom_models.cnn import CNN

import moabb
import mne

moabb.set_log_level('ERROR')
mne.set_log_level('ERROR')

SUBJECTS = 10

physionet_dataset = PhysionetMI()
physionet_dataset.subject_list = physionet_dataset.subject_list[:SUBJECTS]
physionet16_dataset = PhysionetMI16()
physionet16_dataset.subject_list = physionet16_dataset.subject_list[:SUBJECTS]
weibo2014_dataset = Weibo2014()
weibo2014_dataset.subject_list = weibo2014_dataset.subject_list[:SUBJECTS]
weibo2014_16_dataset = Weibo2014_16()
weibo2014_16_dataset.subject_list = weibo2014_16_dataset.subject_list[:SUBJECTS]
assert len(weibo2014_16_dataset.subject_list) == SUBJECTS
assert len(weibo2014_dataset.subject_list) == SUBJECTS
assert len(physionet16_dataset.subject_list) == SUBJECTS
assert len(physionet_dataset.subject_list) == SUBJECTS


datasets = [physionet16_dataset, physionet_dataset, weibo2014_dataset, weibo2014_16_dataset]
dataset_results = {}
dataset_events = ["left_hand", "right_hand", "feet", "hands", "rest"]
sampling = 160 # based on Physionet sampling rate 

electrodes, datasets = find_intersecting_channels(datasets)
print("Datasets used:", [type(d).__name__ for d in datasets])
print("Used electrodes:", electrodes)

paradigm = MotorImagery(n_classes=len(dataset_events), events=dataset_events, resample=sampling)

pipelines = {}

# Base classifiers and preprocessing
svm = OneVsRestClassifier(SVC(kernel='rbf', probability=True))
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)

pipelines['CSP + SVM'] = make_pipeline(csp, svm)
pipelines['CSP + LDA'] = make_pipeline(CSP(n_components=8), LinearDiscriminantAnalysis())

# TGSP (Riemannian) pipeline
pipelines['TGSP + SVM'] = make_pipeline(Covariances("oas"), TangentSpace(metric="riemann"), SVC(kernel="linear", probability=True))

# # Custom CNN pipeline // tensorflow models does not work well with moabb n_jobs>1 - crashes observed due to overallocating memory
# pipelines['CNN'] = CNN(sfreq=sampling)

evaluation = WithinSessionEvaluation(paradigm=paradigm, datasets=datasets, overwrite=True, n_jobs=-1)
evaluation.process(pipelines)
results = evaluation.get_results()

2026-01-31 03:26:43.705930: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-31 03:26:44.155953: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-31 03:26:45.176809: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Searching dataset: PhysionetMI16
Searching dataset: PhysionetMI
Searching dataset: Weibo2014
Searching dataset: Weibo2014_16
Datasets used: ['PhysionetMI16', 'PhysionetMI', 'Weibo2014', 'Weibo2014_16']
Used electrodes: ['CP1', 'FC3', 'CPz', 'FC2', 'CP3', 'Pz', 'FCz', 'C3', 'CP4', 'C2', 'FC4', 'FC1', 'C4', 'CP2', 'Cz', 'C1']


PhysionetMotorImagery16-WithinSession:   0%|          | 0/10 [00:00<?, ?it/s]

No hdf5_path provided, models will not be saved.


/home/mateusz/PROJEKTY/moabb/moabb/datasets/preprocessing.py:278: UserWarning: warnEpochs <Epochs | 29 events (all good), 0 – 3 s (baseline off), ~1.7 MiB, data loaded,
 'left_hand': 8
 'right_hand': 7
 'feet': 0
 'hands': 0
 'rest': 14>
  warn(f"warnEpochs {epochs}")
/home/mateusz/PROJEKTY/moabb/moabb/datasets/preprocessing.py:278: UserWarning: warnEpochs <Epochs | 29 events (all good), 0 – 3 s (baseline off), ~1.7 MiB, data loaded,
 'left_hand': 8
 'right_hand': 7
 'feet': 0
 'hands': 0
 'rest': 14>
  warn(f"warnEpochs {epochs}")
/home/mateusz/PROJEKTY/moabb/moabb/datasets/preprocessing.py:278: UserWarning: warnEpochs <Epochs | 29 events (all good), 0 – 3 s (baseline off), ~1.7 MiB, data loaded,
 'left_hand': 7
 'right_hand': 8
 'feet': 0
 'hands': 0
 'rest': 14>
  warn(f"warnEpochs {epochs}")
/home/mateusz/PROJEKTY/moabb/moabb/datasets/preprocessing.py:278: UserWarning: warnEpochs <Epochs | 29 events (all good), 0 – 3 s (baseline off), ~1.7 MiB, data loaded,
 'left_hand': 0
 'right_

...

Weibo2014_16-WithinSession:  80%|████████  | 8/10 [02:01<00:30, 15.16s/it]/home/mateusz/PROJEKTY/moabb/moabb/datasets/preprocessing.py:278: UserWarning: warnEpochs <Epochs | 400 events (all good), 3 – 7 s (baseline off), ~146.7 MiB, data loaded,
 'left_hand': 80
 'right_hand': 80
 'feet': 80
 'hands': 80
 'rest': 80>
  warn(f"warnEpochs {epochs}")


No hdf5_path provided, models will not be saved.
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 2.9e+02 (2.2e-16 eps * 60 dim * 2.2e+16  max singular value)
    Using tolerance 2.9e+02 (2.2e-16 eps * 60 dim * 2.1e+16  max singular value)
    Using tolerance 2.9e+02 (2.2e-16 eps * 60 dim * 2.1e+16  max singular value)
    Using tolerance 2.9e+02 (2.2e-16 eps * 60 dim * 2.2e+16  max singular value)
    Using tolerance 2.9e+02 (2.2e-16 eps * 60 dim * 2.2e+16  max singular value)
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 compute

Weibo2014_16-WithinSession:  90%|█████████ | 9/10 [02:14<00:14, 14.74s/it]/home/mateusz/PROJEKTY/moabb/moabb/datasets/preprocessing.py:278: UserWarning: warnEpochs <Epochs | 400 events (all good), 3 – 7 s (baseline off), ~146.7 MiB, data loaded,
 'left_hand': 80
 'right_hand': 80
 'feet': 80
 'hands': 80
 'rest': 80>
  warn(f"warnEpochs {epochs}")


No hdf5_path provided, models will not be saved.
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 1.7e+02 (2.2e-16 eps * 60 dim * 1.3e+16  max singular value)
    Using tolerance 1.7e+02 (2.2e-16 eps * 60 dim * 1.2e+16  max singular value)
    Using tolerance 1.7e+02 (2.2e-16 eps * 60 dim * 1.2e+16  max singular value)
    Using tolerance 1.7e+02 (2.2e-16 eps * 60 dim * 1.2e+16  max singular value)
    Using tolerance 1.7e+02 (2.2e-16 eps * 60 dim * 1.2e+16  max singular value)
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 computed from 60 data channels with 0 projectors
    Estimated rank (data): 60
    data: rank 60 compute

Weibo2014_16-WithinSession: 100%|██████████| 10/10 [02:29<00:00, 14.94s/it]


In [2]:
print("Results Summary:")
summary = results.groupby(['pipeline', 'dataset'])['score'].agg(['mean', 'std', 'count'])
summary['mean'] = summary['mean'].round(3)
summary['std'] = summary['std'].round(3)
print(summary.to_string())
print("=" * 50)

print("\nDetailed Results by Subject and Dataset:")
detailed = results.pivot_table(
    index=['dataset', 'subject', 'session'], 
    columns='pipeline', 
    values='score'
)
print(detailed.round(3).to_string())
print("=" * 50)

Results Summary:
                                     mean    std  count
pipeline   dataset                                     
CSP + LDA  PhysionetMotorImagery    0.526  0.171     10
           PhysionetMotorImagery16  0.533  0.121     10
           Weibo2014                0.509  0.122     10
           Weibo2014_16             0.503  0.125     10
CSP + SVM  PhysionetMotorImagery    0.509  0.112     10
           PhysionetMotorImagery16  0.535  0.105     10
           Weibo2014                0.444  0.084     10
           Weibo2014_16             0.455  0.082     10
TGSP + SVM PhysionetMotorImagery    0.638  0.141     10
           PhysionetMotorImagery16  0.539  0.114     10
           Weibo2014                0.715  0.086     10
           Weibo2014_16             0.709  0.089     10

Detailed Results by Subject and Dataset:
pipeline                                 CSP + LDA  CSP + SVM  TGSP + SVM
dataset                 subject session                                  
Physionet